# A/B testing 
Business Question: The company runs an expensive training program for salespeople during the last three months of the year (October–December). Only half of the salespeople are trained. We want to know:<br>
Does assigning customers to trained salespeople increase customer spending enough to justify the cost of the training program?<br>
We randomly assigned customers to trained and untrained sales persons. Customers with `CustomerID % 2 == 0` are assigned to trained customers and customers with `CustomerID % 2 == 1` are assigned to untrained customers.<br>
First, let us read the data, filter customers in October to December 2010 interval, then split them into control and treatment group. 

In [1]:
import os
import pandas as pd
import pandasql as psql
from online_retail.utils.base_funcs import load_data

root_path = os.getcwd()
file_path = os.path.abspath(os.path.join(root_path, '..','..','data/online_retail_II.xlsx'))

data = load_data(file_path=file_path)
data.rename(columns = {'Customer ID':'CustomerID'}, inplace = True)
data.head()


,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,CustomerID,Country
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085.0,United Kingdom
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085.0,United Kingdom
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085.0,United Kingdom


In [4]:
query = '''
SELECT *
FROM data
WHERE CustomerID IS NOT NULL AND Invoice NOT LIKE 'C%'
'''
data_clean = psql.sqldf(query, locals())
data_clean.info()
data_clean.isnull().sum()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 805620 entries, 0 to 805619
Data columns (total 8 columns):
 #   Column       Non-Null Count   Dtype  
---  ------       --------------   -----  
 0   Invoice      805620 non-null  object 
 1   StockCode    805620 non-null  object 
 2   Description  805620 non-null  object 
 3   Quantity     805620 non-null  int64  
 4   InvoiceDate  805620 non-null  object 
 5   Price        805620 non-null  float64
 6   CustomerID   805620 non-null  float64
 7   Country      805620 non-null  object 
dtypes: float64(2), int64(1), object(5)
memory usage: 49.2+ MB


Invoice        0
StockCode      0
Description    0
Quantity       0
InvoiceDate    0
Price          0
CustomerID     0
Country        0
dtype: int64

In [ ]:
query = '''
SELECT CustomerID, SUM(Quantity * Price) AS Monetary
FROM data_clean
WHERE InvoiceDate BETWEEN '2010-10-01' AND '2011-01-01'
GROUP BY CustomerID
'''
testing_group = psql.sqldf(query = query, env = locals())
display(testing_group)

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,CustomerID,Country
0,491725,TEST001,This is a test product.,10,2009-12-14 08:34:00.000000,4.50,12346.0,United Kingdom
1,491742,TEST001,This is a test product.,5,2009-12-14 11:00:00.000000,4.50,12346.0,United Kingdom
2,491744,TEST001,This is a test product.,5,2009-12-14 11:02:00.000000,4.50,12346.0,United Kingdom
3,492718,TEST001,This is a test product.,5,2009-12-18 10:47:00.000000,4.50,12346.0,United Kingdom
4,492722,TEST002,This is a test product.,1,2009-12-18 10:55:00.000000,1.00,12346.0,United Kingdom
5,493410,TEST001,This is a test product.,5,2010-01-04 09:24:00.000000,4.50,12346.0,United Kingdom
6,493412,TEST001,This is a test product.,5,2010-01-04 09:53:00.000000,4.50,12346.0,United Kingdom
7,494450,TEST001,This is a test product.,5,2010-01-14 13:50:00.000000,4.50,12346.0,United Kingdom
8,495295,TEST001,This is a test product.,5,2010-01-22 13:30:00.000000,4.50,12346.0,United Kingdom
9,499763,20682,RED SPOTTY CHILDS UMBRELLA,1,2010-03-02 13:08:00.000000,3.25,12346.0,United Kingdom
